In [1]:
from pathlib import Path

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
)
from conversational_toolkit.embeddings.qwen_vl import Qwen3VLEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import EMBEDDING_MODEL
from loguru import logger
from dotenv import load_dotenv

In [2]:
load_dotenv("../../.env.local")

_DB_DIR = Path("../src/sme_kt_zh_collaboration_rag/db")

IMAGE_VS_PATH = _DB_DIR / "vs_image_new"
TEXT_VS_PATH = _DB_DIR / "vs_text"

In [3]:
logger.info("Initializing vector stores...")
all_chunks = load_chunks(max_files=11)

2026-03-19 17:47:39.680 | INFO     | __main__:<module>:1 - Initializing vector stores...
2026-03-19 17:47:39.682 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:201 - Skipping unsupported file type '': .DS_Store
2026-03-19 17:47:39.682 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:208 - Chunking 10 files from /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data


11


objc[6059]: Class AVFFrameReceiver is implemented in both /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x12b29c3a8) and /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1407403a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[6059]: Class AVFAudioReceiver is implemented in both /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x12b29c3f8) and /Users/pkoerner/Desktop/Kanton_Zurich/rag_venv/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1407403f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
2026-03-19 17:47:44.024 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:227 -   ART_internal_procuremen

In [ ]:
text_chunks = [c for c in all_chunks if c.mime_type.startswith("text/")]
# limit content size to 8k tokens for OpenAI embedding model
text_chunks = [c for c in text_chunks if len(c.content) < 8 * 1024]
await build_vector_store(
    text_chunks,
    OpenAIEmbeddings(model_name=EMBEDDING_MODEL),
    db_path=TEXT_VS_PATH,
    reset=True,
)

In [4]:
# image_chunks = [c for c in all_chunks if c.mime_type.startswith("image/")]
image_chunks = [
    c
    for c in all_chunks
    if c.metadata.get("source_file") == "EPD_cardboard_grupak_corrugated.pdf"
]
await build_vector_store(
    image_chunks,
    Qwen3VLEmbeddings(device="cpu"),
    db_path=IMAGE_VS_PATH,
    reset=True,
    batch_size=1,
)

2026-03-19 17:48:34.323 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:267 - Reset vector store collection at ../src/sme_kt_zh_collaboration_rag/db/vs_image_new
2026-03-19 17:48:34.334 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:275 - Embedding 104 chunks with <conversational_toolkit.embeddings.qwen_vl.Qwen3VLEmbeddings object at 0x144d9d010> ...


AttributeError: 'str' object has no attribute 'mime_type'